# GPT-Neo inference with the HF's Transformers Library
This notebook is a companion of chapter 4 of the "Domain Specific LLMs in Action" book, author Guglielmo Iozzia, [Manning Publications](https://www.manning.com/), 2024.  
The code in this notebook is to introduce readers to the inference (text generation) with the [GPT-Neo model](https://github.com/EleutherAI/gpt-neo) using the Hugging Face's [Transformers library](https://github.com/huggingface/transformers). It can be executed in the Colab free tier with hardware acceleration (GPU).  
More details about the code can be found in the book's chapter.

Install the missing requirements in the Colab VM (HF's Accelerate only).

In [1]:
!pip install -q accelerate
!pip install -q transformers

In [ ]:
import torch


if torch.cuda.is_available():
    print("cuda version:", torch.version.cuda)
    print("torch version:", torch.__version__)
    print("Number of CUDA devices:", torch.cuda.device_count())
    print("Current GPU device:", torch.cuda.current_device())
    print("GPU Name:", torch.cuda.get_device_name(torch.cuda.current_device()))
    print("Total GPU Memory (MB):", round(torch.cuda.get_device_properties(torch.cuda.current_device()).total_memory / 1024**2, 2))
else:
    print("CUDA is not available.")

cuda version: 12.8
torch version: 2.8.0+cu128
Number of CUDA devices: 1
Current GPU device: 0
GPU Name: NVIDIA GeForce RTX 5090
Total GPU Memory (MB): 32109.12


Download the GPT-Neo 2.7B model and the associated tokenizer from the HF's Hub. The model is loaded in full precision and is then loaded into the GPU.

In [2]:
import torch
from transformers import GPTNeoForCausalLM, GPT2Tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_id = "EleutherAI/gpt-neo-2.7B"
tokenizer = GPT2Tokenizer.from_pretrained(model_id)
model = GPTNeoForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
)


Loading weights:   0%|          | 0/420 [00:00<?, ?it/s]

[transformers] GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-2.7B
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
transformer.h.{0...31}.attn.attention.masked_bias | UNEXPECTED |  | 
transformer.h.{0...30}.attn.attention.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
model.to(device)

GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 2560)
    (wpe): Embedding(2048, 2560)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-31): 32 x GPTNeoBlock(
        (ln_1): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): Linear(in_features=2560, out_features=2560, bias=False)
            (v_proj): Linear(in_features=2560, out_features=2560, bias=False)
            (q_proj): Linear(in_features=2560, out_features=2560, bias=False)
            (out_proj): Linear(in_features=2560, out_features=2560, bias=True)
          )
        )
        (ln_2): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
        (mlp): GPTNeoMLP(
          (c_fc): Linear(in_features=2560, out_features=10240, bias=True)
          (c_proj)

Verify where the model layers have been loaded (all in the GPU memory or also RAM and/or disk).

In [ ]:
model.hf_device_map

AttributeError: 'GPTNeoForCausalLM' object has no attribute 'hf_device_map'

Perform standard inference (text completion).

In [4]:
prompt = "The story so far: in the beginning, the universe was created."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

generated_ids = model.generate(input_ids,
                               do_sample=True,
                               temperature=0.9,
                               max_length=200,
                               pad_token_id=50256)
generated_text = tokenizer.decode(generated_ids[0])
print(generated_text)

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


The story so far: in the beginning, the universe was created. Nothing. Literally nothing. Then, the first god was created. His name was Ymir. Ymir was a boy who lived at the bottom of a deep, dark lake. He had an obsession with stones.
Now, here's where things get interesting. Ymir created the first person. She was a young woman who had been thrown out of a cave and left to die. Ymir decided she would be his wife and he would bring her up like a child. He raised her, taught her everything he knew, and loved her the way a mother would a child.

In the mid-90s, a team of scientists discovered the fossil of a woman in a cave in France. This fossil was called Laetoli. The scientists named the woman after the Laetoli National Park in Kenya. It was thought that the woman was the last of the hominid family.
But then, in


Do few-shot text classification (the model can generalize learning from few new and unseen examples.

In [ ]:
prompt = """
Sentence: This movie is very nice.
Sentiment: positive

#####

Sentence: I hated this movie, it sucks.
Sentiment: negative

#####

Sentence: This movie was actually pretty funny.
Sentiment: positive

#####

Sentence: This movie could have been better.
Sentiment: neutral

#####


Sentence: The acting was terrible but the plot was fascinating.
Sentiment:""" 



input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

generated_ids = model.generate(input_ids,
                               do_sample=True,
                               temperature=0.9,
                               max_length=200,
                               pad_token_id=50256)
generated_text = tokenizer.decode(generated_ids[0])
print(generated_text)


Sentence: This movie is very nice.
Sentiment: positive

#####

Sentence: I hated this movie, it sucks.
Sentiment: negative

#####

Sentence: This movie was actually pretty funny.
Sentiment: positive

#####

Sentence: This movie could have been better.
Sentiment: neutral

#####


Sentence: The acting was terrible but the plot was fascinating.
Sentiment: positive

#####

Sentence: They don't make movies like this anymore.
Sentiment: neutral

#####

Sentence: I didn't like this movie at all.
Sentiment: negative

#####

Sentence: This movie was very weird.
Sentiment: neutral

#####

Sentence: Nothing was interesting.
Sentiment: neutral

#####

Sentence: This movie was boring


In [ ]:
def classify_sentiment(sentence, tokenizer, model, device):
    prompt = f"""Sentence: This movie is very nice.
Sentiment: positive

#####

Sentence: I hated this movie, it sucks.
Sentiment: negative

#####

Sentence: This movie was actually pretty funny.
Sentiment: positive

#####

Sentence: This movie could have been better.
Sentiment: neutral

#####

Sentence: {sentence}
Sentiment:"""

    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    generated_ids = model.generate(
        input_ids,
        do_sample=False,       # greedy — deterministic for classification
        max_new_tokens=5,      # only need the label word
        pad_token_id=50256
    )
    # Decode only the newly generated tokens, not the whole prompt
    new_tokens = generated_ids[0][input_ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

sentences = [
    "The acting was terrible but the plot was fascinating.",
    "This movie is amazing and I love it!",
    "This movie is bad and I hate it!",
    "This movie is good and I like it!",
    "This movie is great and I love it!",
]

for s in sentences:
    label = classify_sentiment(s, tokenizer, model, device)
    print(f"Sentence: {s}")
    print(f"Sentiment: {label}")
    print("-----"*5)

Sentence: The acting was terrible but the plot was fascinating.
Sentiment: neutral

#####
-------------------------
Sentence: This movie is amazing and I love it!
Sentiment: positive

#####
-------------------------
Sentence: This movie is bad and I hate it!
Sentiment: negative

#####
-------------------------
Sentence: This movie is good and I like it!
Sentiment: positive

#####
-------------------------
Sentence: This movie is great and I love it!
Sentiment: positive

#####
-------------------------


Do Python code generation.

In [ ]:
prompt = """Instruction: Generate a Python function that lets you reverse a list of integers.

Answer: """
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

generated_ids = model.generate(input_ids,
                               do_sample=True,
                               temperature=0.9,
                               max_length=200,
                               pad_token_id=50256
                               )
generated_text = tokenizer.decode(generated_ids[0])
print(generated_text)

Instruction: Generate a Python function that lets you reverse a list of integers.

Answer: 

 

 **Code Listing 7.10:** 

    >>> def rev(lst):
     ....    return [lst[i-1] for i in range(len(lst))]
     .... 
     >>> rev([5,2,3])
    [[4, 3, 2], [5, 3, 2], [5, 2, 3], [5, 3, 2]]
                         ^
                         |
                   


Do batch text completion.

In [5]:
texts = ["Once there was a man ", "The weather today will be ", "A great soccer player must "]

tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token
encoding = tokenizer(texts, padding=True, return_tensors='pt').to(device)
with torch.no_grad():
    generated_ids = model.generate(**encoding,
                                   do_sample=True,
                                   temperature=0.9,
                                   max_length=50,
                                   pad_token_id=50256)
generated_texts = tokenizer.batch_decode(
    generated_ids, skip_special_tokens=True)

for text in generated_texts:
  print("---------")
  print(text)

---------
Once there was a man 

In the Kingdom of Tarrasch

Who was not, according to the reports

One of the strongest lords of Tarrasch

Who for thirty years held the title

Lord
---------
The weather today will be 
cool and cloudy.  The high temperature 
is expected to be near 90 degrees.  
The low temperature today is expected 
to be in the mid 60s.

High Pressure will continue
---------
A great soccer player must __________ soccer player must __________ soccer player must __________ soccer player must __________ soccer player must __________ soccer player must __________ soccer player must __________ soccer player must __________


Benchmarking the model on text completion: comparing the cases where the KV cache is used to those where it isn't.

In [6]:
import time
import numpy as np

prompt = "The story so far: in the beginning, the universe was created."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

for use_cache in (True, False):
  times = []
  for _ in range(20):
    start = time.time()
    generated_ids = model.generate(input_ids,
                                  do_sample=True,
                                  temperature=0.9,
                                  max_length=200,
                                  pad_token_id=50256,
                                  use_cache=use_cache)
    times.append(time.time() - start)
  print(f"{'Using' if use_cache else 'No'} KV cache: {round(np.mean(times), 3)} +- {round(np.std(times), 3)} seconds")

Using KV cache: 1.675 +- 0.365 seconds
No KV cache: 2.025 +- 0.016 seconds


Benchmarking the model's total generation time.

In [7]:
import time
import numpy as np

prompt = "The story so far: in the beginning, the universe was created."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

max_length = 300
times = []
inference_runs = 21
for _ in range(inference_runs):
  start = time.time()
  generated_ids = model.generate(input_ids,
                                do_sample=True,
                                temperature=0.9,
                                max_length=max_length,
                                pad_token_id=50256,
                                )
  times.append(time.time() - start)
print(f"Average Total Generation time: {round(np.mean(times[1:]), 3)} +- {round(np.std(times[1:]), 3)} seconds")

Average Total Generation time: 2.652 +- 0.379 seconds
